# AI Engine Logic

This notebook builds the AI Engine layer for airport operations: delay prediction, turnaround optimization scoring, resource efficiency, conflict detection, multi-agent resolution, priority scheduling, cost avoidance, past delay learning, and backend-ready dashboard payloads.

## 1. Setup

In [ ]:
import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1200)

now = datetime(2026, 4, 27, 9, 0)


## 2. AI Engine Configuration

In [ ]:
DELAY_CAUSES = ['Gate conflict', 'Fueling overlap', 'Baggage queue', 'Crew positioning', 'Pushback conflict', 'ATC slot miss', 'Weather', 'Maintenance check']
AIRLINES = ['Air India', 'IndiGo', 'SpiceJet', 'Vistara', 'Akasa', 'AirAsia']
AGENTS = {
    'Gate Agent': ['gate_conflict', 'remote_stand', 'terminal_congestion'],
    'Crew Agent': ['crew_shortage', 'fatigue_risk', 'shift_compliance'],
    'Equipment Agent': ['equipment_shortage', 'fuel_low', 'maintenance_risk'],
    'Runway Agent': ['runway_queue', 'slot_pressure', 'closed_runway'],
    'Passenger Agent': ['connection_risk', 'boarding_delay', 'remote_stand'],
}
PENALTY_PER_MIN = {
    'Air India': 520,
    'IndiGo': 460,
    'SpiceJet': 390,
    'Vistara': 540,
    'Akasa': 410,
    'AirAsia': 360,
}

## 3. Generate Historical Operations Data

In [ ]:
def generate_historical_operations(n=950):
    rows = []
    base_time = now - timedelta(days=45)
    for i in range(n):
        ts = base_time + timedelta(minutes=i * 68)
        airline = random.choice(AIRLINES)
        hour = ts.hour
        day_of_week = ts.weekday()
        peak_pressure = 1 if hour in [7, 8, 9, 17, 18, 19] else 0
        weather_risk = np.clip(np.random.beta(2, 8) * 100 + random.choice([0, 0, 15]), 0, 100)
        gate_util = np.clip(np.random.normal(68 + peak_pressure * 18, 14), 20, 100)
        crew_util = np.clip(np.random.normal(72 + peak_pressure * 14, 13), 25, 100)
        equipment_util = np.clip(np.random.normal(70 + peak_pressure * 16, 15), 20, 100)
        runway_util = np.clip(np.random.normal(62 + peak_pressure * 20, 16), 15, 100)
        fatigue_avg = np.clip(np.random.normal(44 + crew_util * 0.28, 10), 10, 95)
        conflict_count = np.random.poisson(0.8 + peak_pressure * 1.2 + (gate_util > 85) + (equipment_util > 85))
        priority_weight = random.choices([0, 8, 15, 25], weights=[72, 12, 12, 4])[0]
        turnaround_target = random.choice([38, 45, 52, 65, 85])

        delay = (
            -8
            + 0.18 * gate_util
            + 0.14 * crew_util
            + 0.12 * equipment_util
            + 0.10 * runway_util
            + 0.16 * weather_risk
            + 3.6 * conflict_count
            + 0.12 * fatigue_avg
            + priority_weight * 0.15
            + np.random.normal(0, 9)
        )
        delay = max(0, round(delay, 1))
        actual_turnaround = turnaround_target + delay * 0.45 + np.random.normal(0, 6)
        ai_intervention = random.random() < (0.35 + min(conflict_count, 4) * 0.09)
        avoided_minutes = round(delay * random.uniform(0.22, 0.62), 1) if ai_intervention else 0
        cause = random.choices(DELAY_CAUSES, weights=[18, 14, 15, 14, 12, 10, 10, 7])[0]

        rows.append({
            'timestamp': ts,
            'flight_id': f'{random.choice(["AI", "6E", "SG", "UK", "QP"])}{random.randint(100, 999)}',
            'airline': airline,
            'hour': hour,
            'day_of_week': day_of_week,
            'peak_pressure': peak_pressure,
            'weather_risk': round(weather_risk, 1),
            'gate_utilization': round(gate_util, 1),
            'crew_utilization': round(crew_util, 1),
            'equipment_utilization': round(equipment_util, 1),
            'runway_utilization': round(runway_util, 1),
            'avg_fatigue': round(fatigue_avg, 1),
            'conflict_count': int(conflict_count),
            'priority_weight': priority_weight,
            'turnaround_target': turnaround_target,
            'actual_turnaround': round(actual_turnaround, 1),
            'delay_minutes': delay,
            'delay_risk_label': int(delay >= 20),
            'delay_cause': cause,
            'ai_intervention': ai_intervention,
            'avoided_minutes': avoided_minutes,
            'penalty_per_min': PENALTY_PER_MIN[airline],
        })
    return pd.DataFrame(rows)


ops_df = generate_historical_operations()
ops_df.head(10)

## 4. Delay Prediction Models

In [ ]:
def add_engineered_features(df):
    enriched = df.copy()
    utilization_cols = ['gate_utilization', 'crew_utilization', 'equipment_utilization', 'runway_utilization']
    enriched['resource_pressure_avg'] = enriched[utilization_cols].mean(axis=1).round(1)
    enriched['resource_pressure_max'] = enriched[utilization_cols].max(axis=1).round(1)
    enriched['crew_fatigue_pressure'] = (enriched['crew_utilization'] * 0.55 + enriched['avg_fatigue'] * 0.45).round(1)
    enriched['weather_conflict_pressure'] = (enriched['weather_risk'] * (1 + enriched['conflict_count'] / 4)).round(1)
    enriched['priority_conflict_pressure'] = (enriched['priority_weight'] * (1 + enriched['peak_pressure'] * 0.8 + enriched['conflict_count'] * 0.15)).round(1)
    return enriched


ops_df = add_engineered_features(ops_df)

FEATURES = [
    'hour', 'day_of_week', 'peak_pressure', 'weather_risk', 'gate_utilization',
    'crew_utilization', 'equipment_utilization', 'runway_utilization', 'avg_fatigue',
    'conflict_count', 'priority_weight', 'turnaround_target', 'resource_pressure_avg',
    'resource_pressure_max', 'crew_fatigue_pressure', 'weather_conflict_pressure',
    'priority_conflict_pressure',
]

indices = np.arange(len(ops_df))
train_idx, test_idx = train_test_split(
    indices,
    test_size=0.25,
    random_state=SEED,
    stratify=ops_df['delay_risk_label'],
)

X_train = ops_df.loc[train_idx, FEATURES]
X_test = ops_df.loc[test_idx, FEATURES]
y_train_reg = ops_df.loc[train_idx, 'delay_minutes']
y_test_reg = ops_df.loc[test_idx, 'delay_minutes']
y_train_cls = ops_df.loc[train_idx, 'delay_risk_label']
y_test_cls = ops_df.loc[test_idx, 'delay_risk_label']

delay_model = RandomForestRegressor(
    n_estimators=240,
    max_depth=12,
    min_samples_leaf=3,
    random_state=SEED,
    n_jobs=1,
)
risk_model = RandomForestClassifier(
    n_estimators=240,
    max_depth=10,
    min_samples_leaf=3,
    class_weight='balanced_subsample',
    random_state=SEED,
    n_jobs=1,
)

delay_model.fit(X_train, y_train_reg)
risk_model.fit(X_train, y_train_cls)

pred_delay = delay_model.predict(X_test)
pred_risk_prob = risk_model.predict_proba(X_test)[:, 1]
pred_risk = (pred_risk_prob >= 0.5).astype(int)

model_metrics = pd.DataFrame([{
    'delay_mae_min': round(mean_absolute_error(y_test_reg, pred_delay), 2),
    'delay_r2': round(r2_score(y_test_reg, pred_delay), 3),
    'risk_accuracy_pct': round(accuracy_score(y_test_cls, pred_risk) * 100, 1),
    'risk_precision_pct': round(precision_score(y_test_cls, pred_risk, zero_division=0) * 100, 1),
    'risk_recall_pct': round(recall_score(y_test_cls, pred_risk, zero_division=0) * 100, 1),
    'risk_f1_pct': round(f1_score(y_test_cls, pred_risk, zero_division=0) * 100, 1),
    'risk_auc_pct': round(roc_auc_score(y_test_cls, pred_risk_prob) * 100, 1),
    'risk_balanced_accuracy_pct': round(balanced_accuracy_score(y_test_cls, pred_risk) * 100, 1),
}])

feature_importance = pd.DataFrame({
    'feature': FEATURES,
    'delay_importance': delay_model.feature_importances_,
    'risk_importance': risk_model.feature_importances_,
})
feature_importance['blended_importance'] = (feature_importance['delay_importance'] * 0.55 + feature_importance['risk_importance'] * 0.45).round(4)
feature_importance = feature_importance.sort_values('blended_importance', ascending=False).reset_index(drop=True)

display(model_metrics)
feature_importance.head(10)


## 5. Live AI Predictions

In [ ]:
def dominant_risk_driver(row):
    drivers = {
        'Weather': row['weather_conflict_pressure'],
        'Gate Pressure': row['gate_utilization'] + row['conflict_count'] * 4,
        'Crew Pressure': row['crew_fatigue_pressure'],
        'Equipment Pressure': row['equipment_utilization'] + row['conflict_count'] * 3,
        'Runway Pressure': row['runway_utilization'] + row['peak_pressure'] * 12,
        'Priority Handling': row['priority_conflict_pressure'],
    }
    return max(drivers, key=drivers.get)


live_df = add_engineered_features(generate_historical_operations(36).tail(36).reset_index(drop=True))
live_feature_frame = live_df[FEATURES]
live_feature_matrix = live_feature_frame.to_numpy()
live_df['predicted_delay_min'] = delay_model.predict(live_feature_frame).round(1)
delay_tree_predictions = np.vstack([tree.predict(live_feature_matrix) for tree in delay_model.estimators_])
live_df['predicted_delay_p10'] = np.quantile(delay_tree_predictions, 0.10, axis=0).round(1)
live_df['predicted_delay_p90'] = np.quantile(delay_tree_predictions, 0.90, axis=0).round(1)
live_df['delay_risk_probability'] = risk_model.predict_proba(live_feature_frame)[:, 1].round(3)
live_df['risk_level'] = pd.cut(live_df['delay_risk_probability'], [-0.01, 0.35, 0.65, 1.0], labels=['Low', 'Medium', 'High'])
live_df['prediction_uncertainty_min'] = (live_df['predicted_delay_p90'] - live_df['predicted_delay_p10']).round(1)
live_df['dominant_risk_driver'] = live_df.apply(dominant_risk_driver, axis=1)
live_df['predicted_penalty_cost'] = (live_df['predicted_delay_min'] * live_df['penalty_per_min']).round(0).astype(int)

live_df[['flight_id', 'airline', 'predicted_delay_min', 'predicted_delay_p90', 'delay_risk_probability', 'risk_level', 'predicted_penalty_cost', 'delay_cause']].sort_values('predicted_delay_min', ascending=False).head(12)


## 6. AI Optimization Scores

In [ ]:
def build_optimization_scores(history, live, metrics):
    avg_turnaround_overrun = np.maximum(history['actual_turnaround'] - history['turnaround_target'], 0).mean()
    turnaround_score = np.clip(100 - avg_turnaround_overrun * 1.2, 0, 100)
    resource_pressure = history[['gate_utilization', 'crew_utilization', 'equipment_utilization']].mean(axis=1)
    resource_efficiency = np.clip(100 - abs(resource_pressure.mean() - 82) * 0.9, 0, 100)
    delay_prevention_rate = np.where(history['delay_minutes'] > 0, history['avoided_minutes'] / history['delay_minutes'].replace(0, np.nan), 0)
    delay_prevention_pct = np.nan_to_num(delay_prevention_rate).mean() * 100
    predictive_accuracy = metrics.iloc[0]['risk_f1_pct'] * 0.55 + metrics.iloc[0]['risk_auc_pct'] * 0.45
    uncertainty_penalty = min(live['prediction_uncertainty_min'].mean() * 1.1, 18)
    parallel_ops = np.clip(100 - live['conflict_count'].mean() * 7 - live['avg_fatigue'].mean() * 0.10 - uncertainty_penalty, 0, 100)

    return pd.DataFrame([
        {'score_name': 'Turnaround Optimization Score', 'value_pct': round(turnaround_score, 1), 'detail': 'Based on actual vs target turnaround overrun'},
        {'score_name': 'Resource Efficiency Score', 'value_pct': round(resource_efficiency, 1), 'detail': 'Balanced gate, crew, and equipment utilization'},
        {'score_name': 'Delay Prevention Rate', 'value_pct': round(delay_prevention_pct, 1), 'detail': 'Avoided minutes as share of potential delay'},
        {'score_name': 'Predictive Accuracy', 'value_pct': round(predictive_accuracy, 1), 'detail': 'F1 and AUC blend on held-out delay risk data'},
        {'score_name': 'Parallel Operations Status', 'value_pct': round(parallel_ops, 1), 'detail': 'Concurrent operation health from live conflicts, fatigue, and uncertainty'},
    ])


scores_df = build_optimization_scores(ops_df, live_df, model_metrics)
scores_df


## 7. Conflict Detection Engine

In [ ]:
def detect_ai_conflicts(live):
    conflicts = []
    for _, row in live.iterrows():
        if row['predicted_delay_min'] >= 28 or row['gate_utilization'] >= 88:
            conflicts.append({'flight_id': row['flight_id'], 'type': 'gate_conflict', 'severity': 'High' if row['predicted_delay_min'] >= 35 else 'Medium', 'agent': 'Gate Agent', 'message': f"Predicted delay {row['predicted_delay_min']} min with gate utilization {row['gate_utilization']}%."})
        if row['crew_utilization'] >= 88 or row['avg_fatigue'] >= 72 or row['dominant_risk_driver'] == 'Crew Pressure':
            conflicts.append({'flight_id': row['flight_id'], 'type': 'crew_shortage', 'severity': 'High' if row['avg_fatigue'] >= 78 else 'Medium', 'agent': 'Crew Agent', 'message': f"Crew pressure {row['crew_utilization']}%, fatigue {row['avg_fatigue']}%."})
        if row['equipment_utilization'] >= 88 or row['dominant_risk_driver'] == 'Equipment Pressure':
            conflicts.append({'flight_id': row['flight_id'], 'type': 'equipment_shortage', 'severity': 'High' if row['prediction_uncertainty_min'] >= 12 else 'Medium', 'agent': 'Equipment Agent', 'message': f"Equipment utilization {row['equipment_utilization']}% near saturation."})
        if row['runway_utilization'] >= 86 or row['weather_risk'] >= 55:
            conflicts.append({'flight_id': row['flight_id'], 'type': 'runway_queue', 'severity': 'High' if row['weather_risk'] >= 70 else 'Medium', 'agent': 'Runway Agent', 'message': f"Runway utilization {row['runway_utilization']}% with weather risk {row['weather_risk']}."})
        if row['priority_weight'] >= 15 and row['delay_risk_probability'] >= 0.5:
            conflicts.append({'flight_id': row['flight_id'], 'type': 'connection_risk', 'severity': 'High', 'agent': 'Passenger Agent', 'message': f"High-priority passenger or connection risk detected ({row['delay_risk_probability']:.2f})."})

    conflicts_df = pd.DataFrame(conflicts)
    if conflicts_df.empty:
        return pd.DataFrame(columns=['conflict_id', 'flight_id', 'type', 'severity', 'agent', 'message'])
    conflicts_df.insert(0, 'conflict_id', [f'CON-{1000 + i}' for i in range(len(conflicts_df))])
    return conflicts_df


conflicts_df = detect_ai_conflicts(live_df)
conflicts_df.head(20)


## 8. Multi-Agent Resolution

In [ ]:
def resolve_conflicts(conflicts):
    if conflicts.empty:
        return pd.DataFrame(columns=['resolution_id', 'conflict_id', 'agent', 'action', 'status', 'estimated_minutes_saved'])

    action_map = {
        'gate_conflict': 'Swap gate with lower priority aircraft or open remote stand plan',
        'crew_shortage': 'Call reserve crew and protect mandatory break compliance',
        'equipment_shortage': 'Reposition nearest idle equipment and defer low-priority task',
        'runway_queue': 'Adjust pushback sequence and move departures to alternate runway',
        'connection_risk': 'Prioritize boarding bridge, baggage transfer, and passenger notification',
    }
    rows = []
    for idx, conflict in conflicts.iterrows():
        minutes_saved = random.randint(4, 22) if conflict['severity'] == 'High' else random.randint(2, 12)
        rows.append({
            'resolution_id': f'RES-{1000 + idx}',
            'conflict_id': conflict['conflict_id'],
            'flight_id': conflict['flight_id'],
            'agent': conflict['agent'],
            'action': action_map.get(conflict['type'], 'Review conflict and notify command center'),
            'status': random.choices(['resolved', 'monitoring', 'queued'], weights=[68, 24, 8])[0],
            'estimated_minutes_saved': minutes_saved,
        })
    return pd.DataFrame(rows)


resolutions_df = resolve_conflicts(conflicts_df)
resolutions_df.head(20)

## 9. Past Delay Learning Feed

In [ ]:
def build_learning_feed(history):
    cause_stats = history.groupby('delay_cause').agg(
        incidents=('flight_id', 'count'),
        avg_delay=('delay_minutes', 'mean'),
        avoided_minutes=('avoided_minutes', 'sum'),
    ).reset_index()
    cause_stats['avg_delay'] = cause_stats['avg_delay'].round(1)
    cause_stats['avoided_minutes'] = cause_stats['avoided_minutes'].round(1)
    cause_stats['learned_pattern'] = cause_stats.apply(
        lambda r: f"{r['delay_cause']} causes {r['avg_delay']} min average delay across {int(r['incidents'])} cases.", axis=1
    )
    cause_stats['recommended_rule'] = cause_stats['delay_cause'].map({
        'Gate conflict': 'Reserve 12-minute gate buffer for peak-hour narrow-body swaps.',
        'Fueling overlap': 'Pre-stage second fuel tractor when fuel utilization exceeds 85%.',
        'Baggage queue': 'Open overflow belt when arrival wave exceeds 3 flights per 30 minutes.',
        'Crew positioning': 'Move reserve crew to next-wave terminal 20 minutes early.',
        'Pushback conflict': 'Sequence pushbacks by runway slot rather than gate readiness alone.',
        'ATC slot miss': 'Escalate flights at T-25 minutes when runway utilization exceeds 80%.',
        'Weather': 'Increase turnaround buffer and protect covered stands during high weather risk.',
        'Maintenance check': 'Assign technician before passenger deboarding for repeat-risk aircraft.',
    })
    return cause_stats.sort_values(['avg_delay', 'incidents'], ascending=False).reset_index(drop=True)


learning_feed_df = build_learning_feed(ops_df)
learning_feed_df

## 10. Cost Avoidance and Priority Rules

In [ ]:
def build_cost_and_rules(history, live, resolutions):
    history = history.copy()
    history['avoided_cost'] = history['avoided_minutes'] * history['penalty_per_min']
    live = live.copy()
    live['risk_cost'] = live['predicted_delay_min'] * live['penalty_per_min']

    cost_summary = pd.DataFrame([{
        'delay_penalty_cost_today': int(live['risk_cost'].sum()),
        'delay_penalty_cost_avoided_history': int(history['avoided_cost'].sum()),
        'estimated_minutes_saved_by_agents': int(resolutions['estimated_minutes_saved'].sum()) if len(resolutions) else 0,
        'resolved_conflicts': int((resolutions['status'] == 'resolved').sum()) if len(resolutions) else 0,
    }])

    rules = pd.DataFrame([
        {'rule_id': 'RULE-001', 'name': 'Emergency flight protection', 'priority': 1, 'condition': 'priority_weight >= 25', 'action': 'Override gate/equipment queue and notify command center'},
        {'rule_id': 'RULE-002', 'name': 'Connection critical protection', 'priority': 2, 'condition': 'priority_weight >= 15 and risk >= medium', 'action': 'Protect boarding bridge and baggage transfer lane'},
        {'rule_id': 'RULE-003', 'name': 'Peak gate buffer', 'priority': 3, 'condition': 'peak_pressure == 1 and gate_utilization > 85', 'action': 'Reserve 12-minute gate clearance buffer'},
        {'rule_id': 'RULE-004', 'name': 'Crew fatigue guardrail', 'priority': 2, 'condition': 'avg_fatigue > 72', 'action': 'Prevent new assignment and call reserve crew'},
        {'rule_id': 'RULE-005', 'name': 'Runway queue guardrail', 'priority': 3, 'condition': 'runway_utilization > 88', 'action': 'Resequence pushback and shift runway where possible'},
    ])
    return cost_summary, rules


cost_summary_df, priority_rules_df = build_cost_and_rules(ops_df, live_df, resolutions_df)
display(cost_summary_df)
priority_rules_df

## 11. AI Engine KPI Snapshot

In [ ]:
def build_ai_kpis(scores, metrics, conflicts, resolutions, cost_summary, live):
    score_map = dict(zip(scores['score_name'], scores['value_pct']))
    high_risk = int((live['risk_level'] == 'High').sum())
    resolved = int((resolutions['status'] == 'resolved').sum()) if len(resolutions) else 0
    resolution_rate = resolved / max(len(resolutions), 1) * 100
    engine_score = np.mean([
        score_map['Turnaround Optimization Score'],
        score_map['Resource Efficiency Score'],
        score_map['Delay Prevention Rate'],
        score_map['Predictive Accuracy'],
        score_map['Parallel Operations Status'],
    ]) - min(high_risk * 1.4, 16) - min(len(conflicts) * 0.35, 14)

    return pd.DataFrame([{
        'turnaround_optimization_score': score_map['Turnaround Optimization Score'],
        'resource_efficiency_score': score_map['Resource Efficiency Score'],
        'delay_prevention_rate_pct': score_map['Delay Prevention Rate'],
        'predictive_accuracy_pct': score_map['Predictive Accuracy'],
        'parallel_operations_score': score_map['Parallel Operations Status'],
        'model_delay_mae_min': metrics.iloc[0]['delay_mae_min'],
        'open_conflicts': len(conflicts),
        'resolved_conflicts': resolved,
        'resolution_rate_pct': round(resolution_rate, 1),
        'high_risk_live_flights': high_risk,
        'delay_penalty_cost_today': int(cost_summary.iloc[0]['delay_penalty_cost_today']),
        'delay_penalty_cost_avoided': int(cost_summary.iloc[0]['delay_penalty_cost_avoided_history']),
        'ai_engine_score': round(float(np.clip(engine_score, 0, 100)), 1),
    }])


kpi_df = build_ai_kpis(scores_df, model_metrics, conflicts_df, resolutions_df, cost_summary_df, live_df)
kpi_df

## 12. AI Engine Visuals

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].barh(scores_df['score_name'], scores_df['value_pct'], color='#7c3aed')
axes[0, 0].set_title('AI Optimization Scores')
axes[0, 0].set_xlim(0, 100)

cause_counts = ops_df['delay_cause'].value_counts().head(8)
axes[0, 1].bar(cause_counts.index, cause_counts.values, color='#0ea5e9')
axes[0, 1].set_title('Top Delay Causes Learned')
axes[0, 1].tick_params(axis='x', rotation=35)

risk_counts = live_df['risk_level'].value_counts().reindex(['Low', 'Medium', 'High'], fill_value=0)
axes[1, 0].bar(risk_counts.index.astype(str), risk_counts.values, color=['#2e7d32', '#f9a825', '#c62828'])
axes[1, 0].set_title('Live Predicted Risk Levels')

agent_counts = conflicts_df['agent'].value_counts() if len(conflicts_df) else pd.Series(dtype=int)
axes[1, 1].barh(agent_counts.index, agent_counts.values, color='#fb923c')
axes[1, 1].set_title('Conflicts by Agent')

plt.tight_layout()
plt.show()

## 13. Backend-Ready Payload

In [ ]:
def build_ai_engine_payload(kpis, metrics, scores, live, conflicts, resolutions, learning, cost_summary, rules, importance):
    live_columns = [
        'flight_id', 'airline', 'predicted_delay_min', 'predicted_delay_p10', 'predicted_delay_p90',
        'delay_risk_probability', 'risk_level', 'prediction_uncertainty_min',
        'dominant_risk_driver', 'predicted_penalty_cost', 'delay_cause',
    ]
    return {
        'generated_at': now.isoformat(),
        'kpis': kpis.iloc[0].to_dict(),
        'model_metrics': metrics.iloc[0].to_dict(),
        'optimization_scores': scores.to_dict(orient='records'),
        'live_predictions': live.sort_values('predicted_delay_min', ascending=False).head(20)[live_columns].assign(risk_level=lambda df: df['risk_level'].astype(str)).to_dict(orient='records'),
        'conflicts': conflicts.to_dict(orient='records') if not conflicts.empty else [],
        'multi_agent_resolution_log': resolutions.to_dict(orient='records') if not resolutions.empty else [],
        'past_delay_learning_feed': learning.head(12).to_dict(orient='records'),
        'cost_summary': cost_summary.iloc[0].to_dict(),
        'priority_rules': rules.to_dict(orient='records'),
        'feature_importance': importance.head(12).to_dict(orient='records'),
    }


payload = build_ai_engine_payload(kpi_df, model_metrics, scores_df, live_df, conflicts_df, resolutions_df, learning_feed_df, cost_summary_df, priority_rules_df, feature_importance)
payload


In [ ]:
print('AI ENGINE SUMMARY')
print('=================')
print(f"Predictive accuracy: {payload['kpis']['predictive_accuracy_pct']}%")
print(f"Delay prevention rate: {payload['kpis']['delay_prevention_rate_pct']}%")
print(f"Open conflicts: {payload['kpis']['open_conflicts']}")
print(f"Resolution rate: {payload['kpis']['resolution_rate_pct']}%")
print(f"Delay penalty cost today: Rs {payload['kpis']['delay_penalty_cost_today']:,}")
print(f"Cost avoided historically: Rs {payload['kpis']['delay_penalty_cost_avoided']:,}")
print(f"AI Engine Score: {payload['kpis']['ai_engine_score']}")
print('\nTop learned rules:')
for item in payload['past_delay_learning_feed'][:5]:
    print(f"- {item['learned_pattern']} Rule: {item['recommended_rule']}")